# 04 - Semantic Search With Provider Comparison

## Learning objectives

- Build a small semantic search pipeline.
- Use OpenAI embeddings for vector retrieval.
- Use Claude and OpenAI for query rewriting and passage explanation.
- Compare provider roles without pretending their APIs are identical.

## Concept primer

Semantic search has two stages: retrieve likely evidence, then use a language model to interpret or explain the result. Embeddings are excellent for the first stage. Generative models are excellent for rewriting messy questions and explaining why a passage is relevant.


## Step 1 - Setup


In [ ]:
# This setup cell keeps imports working whether Jupyter starts in the repo root
# or inside the nlp_transformers_embeddings folder.
from pathlib import Path
import os
import sys

CURRENT = Path.cwd()
COURSE_ROOT = CURRENT.parent if CURRENT.name == "nlp_transformers_embeddings" else CURRENT
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dotenv import load_dotenv
load_dotenv(COURSE_ROOT / ".env", override=False)
load_dotenv(COURSE_ROOT / "nlp_transformers_embeddings" / ".env", override=False)

# Security practice: print whether keys are present, never the key values.
print("LIVE_API enabled:", os.getenv("LIVE_API", "false").lower() == "true")
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("ANTHROPIC_API_KEY loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))


## Step 2 - Prepare chunks and embeddings


In [ ]:
from nlp_transformers_embeddings.utils.mock_data import SAMPLE_DOCUMENTS
from nlp_transformers_embeddings.utils.retrieval import chunk_documents, rank_chunks
from nlp_transformers_embeddings.utils.openai_client import embed_texts, generate_text
from nlp_transformers_embeddings.utils.anthropic_client import rewrite_query, claude_message

chunks = chunk_documents(SAMPLE_DOCUMENTS, chunk_size_words=45, overlap_words=8)
chunk_embeddings = embed_texts([chunk.text for chunk in chunks])

user_question = "Can both OpenAI and Claude create vectors for search?"

# Claude is used to rewrite the query because generation is one of its strengths.
claude_rewritten = rewrite_query(user_question, context_hint="embeddings provider capabilities")
openai_rewritten = generate_text(
    f"Rewrite this as a concise retrieval query: {user_question}",
    instructions="Return only the rewritten query.",
    mock_response="OpenAI Anthropic embeddings provider capabilities vector search",
)

print("Original:", user_question)
print("Claude rewrite:", claude_rewritten)
print("OpenAI rewrite:", openai_rewritten)


## Step 3 - Retrieve evidence

We embed the rewritten query and rank chunks by cosine similarity. In mock mode, the vectors are deterministic but not semantically strong; the pipeline shape is what matters.


In [ ]:
query_embedding = embed_texts([openai_rewritten])[0]
retrieved = rank_chunks(query_embedding, chunk_embeddings, chunks, top_k=3)

for item in retrieved:
    print(f"Rank {item.rank} | score={item.score:.3f} | {item.chunk.title}")
    print(item.chunk.text + "\n")


## Step 4 - Ask both providers to explain relevance

The same retrieved evidence can be interpreted by different generation providers. This is a useful classroom comparison because students see that the retrieval layer and answer layer are separable.


In [ ]:
evidence = "\n\n".join(f"[{item.rank}] {item.chunk.text}" for item in retrieved)
explain_prompt = f"""Question: {user_question}

Retrieved evidence:
{evidence}

Explain which passage is most relevant and why. Keep the answer under 120 words."""

openai_explanation = generate_text(
    explain_prompt,
    instructions="You explain semantic search results for students.",
    mock_response="The provider-differences passage is most relevant because it states that OpenAI has first-party embeddings while Anthropic uses Claude for generation and points to external embedding providers.",
)
claude_explanation = claude_message(
    explain_prompt,
    system="You explain retrieval results clearly and cautiously.",
    mock_response="The strongest passage is the provider capability note. It distinguishes OpenAI embeddings from Claude generation and critique, which directly answers the question.",
)

print("OpenAI explanation:\n", openai_explanation)
print("\nClaude explanation:\n", claude_explanation)


## Student exercise

Ask a question about security, RAG, or transformer attention. Compare the rewritten queries and explanations.

## Debugging checklist

- If live calls do not happen, check `LIVE_API=true` and provider keys.
- If rankings look poor in mock mode, switch to live OpenAI embeddings for a real semantic comparison.
- If a provider explanation invents facts, ask it to cite passage numbers only.

## Production best practices

- Separate retrieval from generation so each layer can be evaluated independently.
- Log original query, rewritten query, retrieved chunk IDs, and final answer.
- Make provider capability differences explicit in architecture docs.

## Reflection questions

1. Why is query rewriting useful before retrieval?
2. Why might two generation providers explain the same evidence differently?
3. Which parts of this pipeline would be replaced by a vector database later?
